In [1]:
# notebook for assignment for homework 2

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
pages = documents[:3]

In [5]:
import json

for page in pages:
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"]
    })

In [6]:
from openai import OpenAI
import os


openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# list models available to your Groq API key
models = openai_client.models.list()
sorted([m.id for m in models.data])[:30]

['allam-2-7b',
 'canopylabs/orpheus-arabic-saudi',
 'canopylabs/orpheus-v1-english',
 'groq/compound',
 'groq/compound-mini',
 'llama-3.1-8b-instant',
 'llama-3.3-70b-versatile',
 'meta-llama/llama-4-scout-17b-16e-instruct',
 'meta-llama/llama-prompt-guard-2-22m',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-120b',
 'openai/gpt-oss-20b',
 'openai/gpt-oss-safeguard-20b',
 'qwen/qwen3-32b',
 'qwen/qwen3.6-27b',
 'whisper-large-v3',
 'whisper-large-v3-turbo']

In [8]:
MODEL="openai/gpt-oss-120b"

In [12]:
from evaluation_utils import Questions, llm_structured
# from openai import OpenAI
# import os


# openai_client = OpenAI(
#     api_key=os.getenv("GROQ_API_KEY"),
#     base_url="https://api.groq.com/openai/v1"
# )

# # list models available to your Groq API key
# models = openai_client.models.list()
# sorted([m.id for m in models.data])[:30]

all_questions = []
token_usage = []

for page in pages:
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"]
    })

    questions, usage = llm_structured(
        client=openai_client,
        model="openai/gpt-oss-20b",
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    token_usage.append(usage.input_tokens)

    for q in questions.questions:
        all_questions.append({
            "question": q,
            "filename": page["filename"]
        })

In [13]:
avg_tokens = sum(token_usage) / len(token_usage)

print(avg_tokens)

1483.0


In [ ]:
# ANS 1 -- 1483

In [14]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [16]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [ ]:
#  Create Search Function

def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        filter_dict={},
        num_results=num_results
    )

In [18]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
